# Notebook 2 — Are forecast intervals trustworthy in production?

**Headline question for a dispatch desk:** Can we trust Week 3 P10–P90 bands as an **80% envelope** for regulating reserve?

This notebook answers that with **conformalized quantile regression (CQR)** on the Week 3 quantile LightGBM stack:

1. **Question** — raw quantile coverage vs nominal
2. **Method** — split conformal + CQR with chronological calibration
3. **Fit** — quantile LightGBM (P05/P10/P50/P90/P95)
4. **Raw coverage** — empirical P10–P90 and P05–P95 on a held-out test block
5. **MAPIE CQR** — conformal correction at 80% and 90%
6. **Numbers** — coverage table + fan plot
7. **Dispatch read** — under-calibrated bands under-procure reserve
8. **Long rolling backtest** — expanding-window CQR over ~365 days; coverage-over-time chart
9. **By month and regime** — coverage, width, pinball sharpness
10. **Market terms** — what guaranteed coverage means for reserve and imbalance

See also: [`docs/conformal_mental_model.md`](../docs/conformal_mental_model.md)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.conformal.backtest import (
    RollingOriginConfig,
    coverage_by_month,
    coverage_by_regime,
    pinball_comparison,
    rolling_origin_backtest,
)
from src.conformal.cqr import run_conformal_cqr
from src.conformal.quantile_lgbm import WEEK3_MODEL_PARAMS, QuantileLGBM
from src.conformal.simulate import WindSimulationConfig, feature_columns, simulate_wind_forecast
from src.conformal.split import chronological_conformal_split

## 1. Question

Week 3 trains separate LightGBM models at **P10, P50, P90** with pinball loss. Default models often **under-cover** (~59% empirical vs 80% nominal on June 2019 CV).

A TSO sizing reserve from P10–P90 needs **empirical coverage**, not just a label. Conformal prediction widens under-covering bands until marginal coverage matches nominal — with no distributional assumption.

**Fine print:** guarantees need approximate **exchangeability**. We use a **chronological** train → calibration → test split (24 h gap), not a random shuffle.

## 2. Data and split

Synthetic DE-style day-ahead wind (~120 days) with heteroskedastic tails — the Week 3 under-coverage story in miniature. If a sibling `wind-quantile-forecast/data/processed/day_ahead_wind.parquet` exists locally, swap it in here.

In [ ]:
wind_parquet = (
    Path("..") / ".." / "wind-quantile-forecast" / "data" / "processed" / "day_ahead_wind.parquet"
)

if wind_parquet.exists():
    frame = pd.read_parquet(wind_parquet)
    print(f"Loaded Week 3 parquet: {len(frame):,} rows")
else:
    frame = simulate_wind_forecast(WindSimulationConfig(n_days=120, seed=7))
    print(f"Using synthetic wind: {len(frame):,} hourly rows")

cols = feature_columns()
train, cal, test = chronological_conformal_split(frame, gap_hours=24)
print(f"Train={len(train):,}  Cal={len(cal):,}  Test={len(test):,}")

## 3. Fit quantile LightGBM + MAPIE CQR

Port of Week 3 `QuantileGBM` (LightGBM only) with locked hyperparameters from `final_model_params.json`.

In [ ]:
model = QuantileLGBM(model_params=WEEK3_MODEL_PARAMS)
result = run_conformal_cqr(train, cal, test, cols, model=model)

## 4. First coverage numbers

Compare **raw quantile-model intervals** vs **conformalized (CQR)** on the held-out test block.

In [ ]:
rows = [
    {
        "interval": "Raw P10–P90",
        "nominal": "80%",
        "coverage": result.raw_80.coverage,
        "gap": result.raw_80.coverage_gap,
        "mean_width_mw": result.raw_80.mean_width,
    },
    {
        "interval": "CQR 80%",
        "nominal": "80%",
        "coverage": result.cqr_80.coverage,
        "gap": result.cqr_80.coverage_gap,
        "mean_width_mw": result.cqr_80.mean_width,
    },
    {
        "interval": "Raw P05–P95",
        "nominal": "90%",
        "coverage": result.raw_90.coverage,
        "gap": result.raw_90.coverage_gap,
        "mean_width_mw": result.raw_90.mean_width,
    },
    {
        "interval": "CQR 90%",
        "nominal": "90%",
        "coverage": result.cqr_90.coverage,
        "gap": result.cqr_90.coverage_gap,
        "mean_width_mw": result.cqr_90.mean_width,
    },
]
coverage_table = pd.DataFrame(rows)
coverage_table

**Typical finding:** raw quantiles **under-cover**; CQR hits nominal (± finite-sample noise) by **widening** intervals.

In [ ]:
plot_df = result.test_frame.sort_values("valid_time").head(168)
t = plot_df["valid_time"]
y = plot_df["wind_mw"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(t, plot_df["raw_p10"], plot_df["raw_p90"], alpha=0.25, label="Raw P10–P90")
ax.fill_between(t, plot_df["cqr80_lo"], plot_df["cqr80_hi"], alpha=0.25, label="CQR 80%")
ax.plot(t, y, color="black", linewidth=0.8, label="Actual")
ax.plot(t, plot_df["pred_p50"], color="C0", linewidth=0.8, label="P50")
ax.set_ylabel("Wind (MW)")
ax.set_title("One week: raw vs conformalized 80% envelope")
ax.legend(loc="upper right")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Dispatch read (single split)

| Finding | Operational impact |
|---|---|
| Raw P10–P90 under-covers | Reserve buffer sized from the label is **too small** — more balancing activation and imbalance exposure |
| CQR hits ~80% / ~90% | Bands are **honest** for the calibration window — wider, but trustworthy |
| Wider CQR intervals | Dispatchers hold **more flex** around P50 in risky hours |

Week 3 pinball tuning optimizes **quantile loss**, not **coverage**. CQR is the distribution-free layer that closes the gap.

The sections below run a **long rolling backtest** — the production read a desk would actually monitor.

## 6. Long rolling backtest

Rolling-origin evaluation over the **longest window the data allows**: expanding train, fixed calibration and test blocks (30-day step), 24 h gaps between blocks.

Synthetic wind uses **365 days** with seasonal heteroskedasticity and a mid-sample residual-scale jump so coverage can drift by month — the story a reserve desk needs to see, not just one held-out slice.

In [ ]:
if wind_parquet.exists():
    long_frame = pd.read_parquet(wind_parquet)
    print(f"Long backtest on Week 3 parquet: {len(long_frame):,} rows")
else:
    long_frame = simulate_wind_forecast(
        WindSimulationConfig(
            n_days=365,
            seed=7,
            seasonal_heteroskedasticity=True,
            drift_day=180,
            drift_scale_factor=1.35,
        )
    )
    print(f"Long backtest on synthetic wind: {len(long_frame):,} hourly rows")

backtest_cfg = RollingOriginConfig()
FAST_PARAMS = {"n_estimators": 80, "num_leaves": 65, "verbosity": -1}

backtest = rolling_origin_backtest(
    long_frame,
    cols,
    config=backtest_cfg,
    model_params=FAST_PARAMS,
)
print(f"Folds: {len(backtest.fold_table)}")
backtest.fold_table[
    [
        "fold_id",
        "test_month",
        "raw_80_coverage",
        "cqr_80_coverage",
        "raw_80_width",
        "cqr_80_width",
        "n_test",
    ]
]

## 7. Coverage over time — the money chart

Each point is one rolling test block. A desk monitors this: **does empirical coverage track the nominal line month after month?** Raw quantiles that sag below 80% are unpriced shortage risk.

In [ ]:
fold_plot = backtest.fold_table.sort_values("test_month")

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(
    fold_plot["test_month"],
    fold_plot["raw_80_coverage"],
    marker="o",
    label="Raw P10–P90",
    color="C1",
)
ax.plot(
    fold_plot["test_month"],
    fold_plot["cqr_80_coverage"],
    marker="s",
    label="CQR 80%",
    color="C0",
)
ax.axhline(0.80, color="0.3", linestyle="--", linewidth=1, label="Nominal 80%")
ax.axhline(0.90, color="0.5", linestyle=":", linewidth=1, label="Nominal 90% (reference)")
ax.set_ylabel("Empirical coverage")
ax.set_xlabel("Test block start month")
ax.set_title("Rolling-origin coverage over time")
ax.set_ylim(0.45, 1.02)
ax.legend(loc="lower left")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Coverage and width by month / wind regime

Aggregate over all rolling test hours. **Regime** = tercile of NWP hub wind speed (low / mid / high) — the hours where reserve sizing hurts most.

In [ ]:
month_table = coverage_by_month(backtest.predictions).round(3)
regime_table = coverage_by_regime(backtest.predictions).round(3)
print("Coverage by calendar month:")
month_table

In [ ]:
print("Coverage by wind regime:")
regime_table

## 9. Pinball loss — sharpness was not destroyed

CQR widens intervals for **coverage**, not to improve pinball. P50 pinball should be unchanged (same median model). Tail pinball on CQR bounds is often *worse* — that is expected; we bought honesty, not tighter quantiles.

In [ ]:
pinball_table = pinball_comparison(backtest.predictions)
pinball_table.round(1)

## 10. What this means in market terms

**Guaranteed-coverage intervals are what a trading desk or reserve-planning team can actually contract against.**

A model that claims **80%** but delivers **~65%** creates real financial exposure:

| Risk | Mechanism |
|---|---|
| **Under-procured reserve** | P10–P90 sized as an 80% envelope but covering only ~65% of hours → more redispatch and balancing activation |
| **Unpriced imbalance** | Actual wind outside the band more often than the label implies → settlement exposure on the uncovered tail |
| **False confidence in limits** | Traders or optimizers treat the band as a hard constraint; under-coverage breaks that constraint silently |

**CQR** pays in **width** (more MW held around P50) and buys a label you can write into a reserve or imbalance policy. It does not promise the narrowest band — it promises the **stated coverage level** holds on the calibration window (and rolling re-calibration tracks slow drift better than one split).

Illustrative scaling (not a full cost model):

- Reserve slice ≈ `(P90 − P10) / 2` MW per hour (half the 80% band)
- If raw bands under-cover by **15 percentage points**, roughly **15% more hours** fall outside the band than planned
- At **500 MW** mean reserve and **€120/MWh** imbalance premium for those surprise hours → **€9,000/hour** of unplanned exposure per 100 MW of reserve error (order-of-magnitude illustration)

**Decision read:** monitor the **coverage-over-time chart** the way Notebook 1 monitors placebo p-values. If raw coverage sags in winter or high-wind regimes, do **not** size regulating reserve from the raw quantile label alone — conformalize or hold extra flex explicitly.

## 11. What would break

- **Exchangeability under drift** — rolling calibration is a practical patch, not a proof; storm clusters and NWP/fleet changes still violate the fine print.
- **Conditional coverage** — marginal 80% does not guarantee 80% inside every high-wind hour; check regime tables.
- **Feature pipeline drift** — CQR corrects **marginal** coverage, not bias from stale NWP or broken lags.
- **Synthetic data** — numbers here illustrate the method; re-run on real Week 3 parquet for production claims.
- **EnbPI / ACI** — online adaptive conformal methods remain a future increment if coverage drifts faster than monthly re-calibration can track.